# Siamese Network for Person Re-Identification

GPU training notebook for the GitHub project. Heavy training is intended to run on Google Colab; the trained checkpoint can then be used for CPU inference locally.


## 1. Enable GPU

In Colab: **Runtime → Change runtime type → GPU**.


In [ ]:
!nvidia-smi


## 2. Get the repository

After publishing the project to GitHub, replace the placeholder URL below with your repository URL.


In [ ]:
# Replace with your actual GitHub repository URL
REPO_URL = "https://github.com/YOUR_USERNAME/siamese-person-reid.git"
!git clone $REPO_URL
%cd siamese-person-reid


In [ ]:
!pip install -q -r requirements.txt


## 3. Mount Google Drive

Store the dataset and trained checkpoint in Drive so they persist after the Colab runtime ends.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 4. Set the dataset path

The project expects `train/train.csv` and the corresponding images under `train/`.


In [ ]:
DATA_ROOT = "/content/drive/MyDrive/person-reid-dataset"

# Expected:
# DATA_ROOT/train/train.csv
# DATA_ROOT/train/*.jpg

!find "$DATA_ROOT" -maxdepth 2 -type f | head -20


## 5. Train on GPU


In [ ]:
!python -m src.train --data-root "$DATA_ROOT" --device cuda --seed 42


## 6. Copy the best checkpoint to Google Drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/siamese-person-reid/checkpoints"
!cp checkpoints/best_model.pt "/content/drive/MyDrive/siamese-person-reid/checkpoints/best_model.pt"


## 7. Generate the embedding database


In [ ]:
!python -m src.embeddings \
  --checkpoint checkpoints/best_model.pt \
  --triplet-csv "$DATA_ROOT/train/train.csv" \
  --image-dir "$DATA_ROOT/train" \
  --output outputs/results/database.csv \
  --device cuda


## 8. Optional retrieval evaluation

Run this only after generating the database. The identity extraction assumes the filename convention used by the original dataset.


In [ ]:
!python -m src.evaluate \
  --database outputs/results/database.csv \
  --top-k 1 5 10
